# Phase 3: Transfer Learning Model Architecture & Training
## Project: Indian Medicinal Plant Identification using Deep Learning & XAI

This notebook outlines the Deep Learning transfer learning methodology using **MobileNetV3-Large / EfficientNet-B0 / ResNet-50** with hardware acceleration on NVIDIA RTX 3050 GPU.

### Training Strategy:
- **Phase 1 (Warmup):** Freeze backbone weights, train customized 93-class classifier head with Label Smoothing Cross-Entropy.
- **Phase 2 (Fine-Tuning):** Unfreeze top convolutional blocks with reduced learning rate ($10^{-4}$).
- **Optimization:** AdamW with weight decay, mixed precision (`torch.amp`), and `ReduceLROnPlateau` scheduling.


In [ ]:
import sys, os
sys.path.append(r"d:\mainpojectD7")
import torch
import torch.nn as nn
from src.model import build_model

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

model = build_model(num_classes=93, backbone_name="mobilenet_v3_large", pretrained=True, device=device)
print(f"Total model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


In [ ]:
# Display architecture summary
print("Classification Head Architecture:")
print(model.classifier)
print("\nTarget Convolutional Layer for Grad-CAM:")
print(model.get_gradcam_target_layer())


In [ ]:
# Display Training Curves from completed runs
import json
import matplotlib.pyplot as plt

history_file = r"d:\mainpojectD7\saved_models\training_history.json"
if os.path.exists(history_file):
    with open(history_file) as f:
        hist = json.load(f)
    
    epochs = range(1, len(hist['train_loss']) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    
    ax1.plot(epochs, hist['train_loss'], 'o-', label='Train Loss', color='#10b981')
    ax1.plot(epochs, hist['val_loss'], 's--', label='Val Loss', color='#ef4444')
    ax1.set_title("Training & Validation Loss")
    ax1.set_xlabel("Epoch")
    ax1.legend()
    
    ax2.plot(epochs, [a*100 for a in hist['train_acc']], 'o-', label='Train Acc (%)', color='#10b981')
    ax2.plot(epochs, [a*100 for a in hist['val_acc']], 's--', label='Val Acc (%)', color='#3b82f6')
    ax2.set_title("Classification Accuracy (%)")
    ax2.set_xlabel("Epoch")
    ax2.legend()
    
    plt.tight_layout()
    plt.show()
else:
    print("Training history will appear once model training finishes.")
